# Calculate lambdas 
1. Calculate the lambdas for the studies
2. Calculate the lambdas for the metas

In [3]:
## Import the necessary packages 
import os
import numpy as np
import pandas as pd
import math
import sys
import subprocess
import statsmodels.api as sm
import scipy
from scipy import stats
from scipy.stats import chi2
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import argparse
from scipy.stats import chi2

## Print out package versions
## Getting packages loaded into this notebook and their versions to allow for reproducibility
import pkg_resources
import types
from datetime import date

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

## Define function 
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            name = val.__name__.split(".")[0]
        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages:
            name = poorly_named_packages[name]

        yield name

## Get a list of packages imported 
imports = list(set(get_imports()))

requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name != "pip":
        requirements.append((m.project_name, m.version))

## Print out packages and versions 
print(f"PACKAGE VERSIONS ({date})")
for r in requirements:
    print("\t{}=={}".format(*r))

## Also print which Python is being used
print("\nPYTHON INFO")
print(f"\tPython executable: {sys.executable}")

/tmp/ipykernel_1766084/2774862509.py:20: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


PACKAGE VERSIONS (21-NOV-2025)
	matplotlib==3.8.4
	numpy==1.26.4
	pandas==2.2.3
	scipy==1.13.1
	seaborn==0.13.2
	statsmodels==0.14.4

PYTHON INFO
	Python executable: /usr/local/apps/python/py3.11/bin/python


In [4]:
def compute_lambda(
    path,
    cohort_name,
    p_col,
    maf_col,
    sep="\t",
    maf_thresh=0.05,
    n_cases=None,
    n_controls=None,
    n_total=None,
):
    """
    Compute lambda_GC and lambda_1000 from summary statistics file.
    
    Parameters
    ----------
    path : str
        Path to summary stats file
    cohort_name : str
        Name of cohort
    p_col : str
        Column with p-values
    maf_col : str
        Column with MAF
    maf_thresh : float
        Minimum MAF threshold (default 0.05)
    n_cases, n_controls : float
        For case-control studies
    n_total : float
        Total N (overrides n_cases/n_controls)
    """
    # Read and filter data
    df = pd.read_csv(path, sep=sep, compression="infer")
    df = df[[p_col, maf_col]].dropna()
    df = df[df[maf_col] >= maf_thresh]
    
    # Get valid p-values and convert to chi-square
    p = pd.to_numeric(df[p_col], errors="coerce")
    p = p[(p > 0) & (p < 1)].values
    p = np.clip(p, 1e-300, 1.0)  # Floor for ultra-small p-values
    
    chisq = chi2.isf(p, df=1)
    chisq = chisq[np.isfinite(chisq) & (chisq >= 0)]
    
    # Compute lambda_GC
    lambda_gc = np.median(chisq) / 0.4549364  # Expected median for df=1
    
    # Compute effective N and lambda_1000
    if n_total is not None:
        neff = float(n_total)
    elif n_cases and n_controls:
        neff = 4.0 / (1.0/n_cases + 1.0/n_controls)
    else:
        neff = None
    
    lambda_1000 = 1.0 + (lambda_gc - 1.0) * (1000.0 / neff) if neff else None
    
    # Return results
    return pd.DataFrame([{
        "cohort": cohort_name,
        "n_cases": n_cases,
        "n_controls": n_controls,
        "neff": neff,
        "lambda_gc": round(lambda_gc, 4),
        "lambda_1000": round(lambda_1000, 4) if lambda_1000 else None,
        "n_snps": len(chisq)
    }])



# Per Cohort

In [10]:
## GP2 AAC 
gp2_aac = compute_lambda(
    path=f"{WORK_DIR}/data/GP2_R11/AAC/GP2_AAC_GWAS_R11.wAlleles.FOR_PLINK.txt",
    cohort_name="GP2 AAC R11",
    p_col="P",
    maf_col="EAF",
    maf_thresh=0.05,
    n_cases=472,
    n_controls=807,
)

gp2_aac

,cohort,n_cases,n_controls,neff,lambda_gc,lambda_1000,n_snps
0,GP2 AAC R11,472,807,1191.255668,1.0184,1.0154,8019931


In [11]:
## GP2 AFR 
gp2_afr = compute_lambda(
    path=f"{WORK_DIR}/data/GP2_R11/AFR/GP2_AFR_GWAS_R11.wAlleles.FOR_PLINK.txt",
    cohort_name="GP2 AFR R11",
    p_col="P",
    maf_col="EAF",
    maf_thresh=0.05,
    n_cases=2504,
    n_controls=4198,
)

gp2_afr

,cohort,n_cases,n_controls,neff,lambda_gc,lambda_1000,n_snps
0,GP2 AFR R11,2504,4198,6273.823933,1.0974,1.0155,8400802


In [12]:
## 23andMe AAC 
aac_23andme = compute_lambda(
    path=f"{WORK_DIR}/data/23andMe/23andMe_AAC.wAlleles.FOR_PLINK.txt",
    cohort_name="23andMe AAC",
    p_col="P",
    maf_col="EAF",
    maf_thresh=0.05,
    n_cases=288,
    n_controls=193985,
)

aac_23andme

,cohort,n_cases,n_controls,neff,lambda_gc,lambda_1000,n_snps
0,23andMe AAC,288,193985,1150.292218,1.0347,1.0301,15543489


In [13]:
## MVP AAC 
aac_mvp = compute_lambda(
    path=f"{WORK_DIR}/data/MVP/MVP_AAC.wAlleles.FOR_PLINK.txt",
    cohort_name="MVP AAC",
    p_col="P",
    maf_col="EAF",
    maf_thresh=0.05,
    n_cases=711,
    n_controls=120893,
)

aac_mvp

,cohort,n_cases,n_controls,neff,lambda_gc,lambda_1000,n_snps
0,MVP AAC,711,120893,2827.371567,1.0094,1.0033,9108110


# Meta-analyses

In [19]:
## AAC Only 

aac_meta = compute_lambda(
    path=f"{WORK_DIR}/results/AAC_META_GP2_23andMe_MVP/GP2_R11_AAC_23andMe_MVP_GWAS.wAlleles.METAL_meta1.meta",
    cohort_name="AAC Only Meta: GP2 AAC / 23andMe AAC / MVP AAC",
    p_col="P-value",
    maf_col="Freq1",
    maf_thresh=0.05,
    n_cases=1471,
    n_controls=315685,
)

aac_meta

,cohort,n_cases,n_controls,neff,lambda_gc,lambda_1000,n_snps
0,AAC Only Meta: GP2 AAC / 23andMe AAC / MVP AAC,1471,315685,5856.709443,1.0202,1.0035,17735708


In [7]:
## AFR/AAC 

combined_meta = compute_lambda(
    path=f"{WORK_DIR}/results/META_GP2_23andMe_MVP/GP2_R11_AAC_AFR_23andMe_MVP_GWAS.wAlleles.METAL_meta1.meta",
    cohort_name="AFR/AAC Meta-analysis: GP2 AAC / GP2 AFR / 23andMe AAC / MVP AAC",
    p_col="P-value",
    maf_col="Freq1",
    maf_thresh=0.05,
    n_cases=3975,
    n_controls=319883,
)

combined_meta

,cohort,n_cases,n_controls,neff,lambda_gc,lambda_1000,n_snps
0,AFR/AAC Meta-analysis: GP2 AAC / GP2 AFR / 23a...,3975,319883,15704.845025,1.0625,1.004,18042490
